# AI-Powered Data Quality & Cleaning Engine
**Stack:** Python, Pandas, Gemini API  
**Dataset:** Synthetic retail orders with intentional data-quality issues.


## 0. Install dependencies
Run once if packages are missing.

In [1]:
%pip install -q pandas google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports and paths

In [4]:
from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
INPUT_DIR = PROJECT_DIR / "input"
OUTPUT_DIR = PROJECT_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

input_file = INPUT_DIR / "raw_orders.csv"
print(input_file.exists(), input_file)

True C:\jn\month 1 project\AI_Data_Quality_Engine\input\raw_orders.csv


## 2. Load the raw file
Read all source values as strings to preserve their original representation before conversion.

In [5]:
raw_df = pd.read_csv(input_file, dtype="string")
print("Shape:", raw_df.shape)
display(raw_df.head())

Shape: (253, 7)


,order_id,customer_id,order_date,quantity,unit_price,sales_region,payment_status
0,ORD0001,C120,2026-01-01,5,249,North,Pending
1,ORD0002,C123,2026-01-22,2,999,West,Completed
2,ORD0003,C100,2026-01-07,4,999,south,Completed
3,ORD0004,C117,2026-01-23,7,249,West,PENDING
4,ORD0005,C108,2026-01-25,3,1499,West,Cancelled


## 3. Profile the source data
This stage observes the file; it does not clean it.

In [6]:
print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())
print("Data types:\\n", raw_df.dtypes)
print("Missing values:\\n", raw_df.isna().sum())
print("Exact duplicate rows:", raw_df.duplicated().sum())

Shape: (253, 7)
Columns: ['order_id', 'customer_id', 'order_date', 'quantity', 'unit_price', 'sales_region', 'payment_status']
Data types:\n order_id          string[python]
customer_id       string[python]
order_date        string[python]
quantity          string[python]
unit_price        string[python]
sales_region      string[python]
payment_status    string[python]
dtype: object
Missing values:\n order_id           0
customer_id       10
order_date         0
quantity           0
unit_price         6
sales_region      41
payment_status    41
dtype: int64
Exact duplicate rows: 2


## 4. Create a working copy and standardize text

In [7]:
work_df = raw_df.copy()

text_columns = ["order_id", "customer_id", "sales_region", "payment_status"]
for column in text_columns:
    work_df[column] = work_df[column].astype("string").str.strip()

work_df["sales_region"] = work_df["sales_region"].str.title()
work_df["payment_status"] = work_df["payment_status"].str.title()

display(work_df.head())

,order_id,customer_id,order_date,quantity,unit_price,sales_region,payment_status
0,ORD0001,C120,2026-01-01,5,249,North,Pending
1,ORD0002,C123,2026-01-22,2,999,West,Completed
2,ORD0003,C100,2026-01-07,4,999,South,Completed
3,ORD0004,C117,2026-01-23,7,249,West,Pending
4,ORD0005,C108,2026-01-25,3,1499,West,Cancelled


## 5. Convert dates and numeric columns
Invalid conversions become missing values, which we will flag in validation.

In [8]:
work_df["order_date"] = pd.to_datetime(work_df["order_date"], errors="coerce")
work_df["quantity"] = pd.to_numeric(work_df["quantity"], errors="coerce")
work_df["unit_price"] = pd.to_numeric(work_df["unit_price"], errors="coerce")

print(work_df.dtypes)
display(work_df.head())

order_id          string[python]
customer_id       string[python]
order_date        datetime64[ns]
quantity                   Int64
unit_price                 Int64
sales_region      string[python]
payment_status    string[python]
dtype: object


,order_id,customer_id,order_date,quantity,unit_price,sales_region,payment_status
0,ORD0001,C120,2026-01-01,5,249,North,Pending
1,ORD0002,C123,2026-01-22,2,999,West,Completed
2,ORD0003,C100,2026-01-07,4,999,South,Completed
3,ORD0004,C117,2026-01-23,7,249,West,Pending
4,ORD0005,C108,2026-01-25,3,1499,West,Cancelled


## 6. Apply validation rules
These are illustrative business rules: required IDs, valid date, positive whole quantity, positive price, required region/status, and duplicate handling.

In [9]:
work_df["missing_order_id"] = work_df["order_id"].isna()
work_df["missing_customer_id"] = work_df["customer_id"].isna()
work_df["invalid_order_date"] = work_df["order_date"].isna()
work_df["invalid_quantity"] = (
    work_df["quantity"].isna()
    | (work_df["quantity"] <= 0)
    | (work_df["quantity"] % 1 != 0)
)
work_df["invalid_unit_price"] = (
    work_df["unit_price"].isna()
    | (work_df["unit_price"] <= 0)
)
work_df["missing_sales_region"] = work_df["sales_region"].isna()
work_df["missing_payment_status"] = work_df["payment_status"].isna()
work_df["exact_duplicate"] = raw_df.duplicated(keep="first")

display(work_df.head())

,order_id,customer_id,order_date,quantity,unit_price,sales_region,payment_status,missing_order_id,missing_customer_id,invalid_order_date,invalid_quantity,invalid_unit_price,missing_sales_region,missing_payment_status,exact_duplicate
0,ORD0001,C120,2026-01-01,5,249,North,Pending,False,False,False,False,False,False,False,False
1,ORD0002,C123,2026-01-22,2,999,West,Completed,False,False,False,False,False,False,False,False
2,ORD0003,C100,2026-01-07,4,999,South,Completed,False,False,False,False,False,False,False,False
3,ORD0004,C117,2026-01-23,7,249,West,Pending,False,False,False,False,False,False,False,False
4,ORD0005,C108,2026-01-25,3,1499,West,Cancelled,False,False,False,False,False,False,False,False


## 7. Build rejection reasons
A row may have multiple reasons. The reasons remain available for audit.

In [10]:
reason_columns = [
    "missing_order_id", "missing_customer_id", "invalid_order_date",
    "invalid_quantity", "invalid_unit_price", "missing_sales_region",
    "missing_payment_status", "exact_duplicate"
]

work_df["rejection_reason"] = ""
for column in reason_columns:
    reason_name = column.replace("_", " ")
    work_df.loc[work_df[column], "rejection_reason"] = (
        work_df["rejection_reason"] + reason_name + "; "
    )

work_df["rejected"] = work_df["rejection_reason"] != ""
work_df["record_status"] = "Valid"
work_df.loc[work_df["rejected"], "record_status"] = "Rejected"

print(work_df["record_status"].value_counts())
display(work_df[["order_id", "record_status", "rejection_reason"]].head(12))

record_status
Valid       149
Rejected    104
Name: count, dtype: int64


,order_id,record_status,rejection_reason
0,ORD0001,Valid,
1,ORD0002,Valid,
2,ORD0003,Valid,
3,ORD0004,Valid,
4,ORD0005,Valid,
5,ORD0006,Valid,
6,ORD0007,Valid,
7,ORD0008,Valid,
8,ORD0009,Rejected,missing payment status;
9,ORD0010,Valid,


## 8. Separate valid and rejected records

In [11]:
valid_df = work_df.loc[work_df["rejected"] == False].copy()
rejected_df = work_df.loc[work_df["rejected"] == True].copy()

print("Source:", len(raw_df))
print("Valid:", len(valid_df))
print("Rejected:", len(rejected_df))
print("Reconciliation:", len(raw_df) == len(valid_df) + len(rejected_df))

Source: 253
Valid: 149
Rejected: 104
Reconciliation: True


## 9. Build an aggregate quality report
Only aggregate counts are sent to Gemini; no customer-level rows are sent.

In [12]:
rejection_counts = {}
for column in reason_columns:
    rejection_counts[column] = int(work_df[column].sum())

quality_summary = {
    "source_file": input_file.name,
    "total_records": int(len(raw_df)),
    "valid_records": int(len(valid_df)),
    "rejected_records": int(len(rejected_df)),
    "rejection_rate_percent": round(len(rejected_df) / len(raw_df) * 100, 2) if len(raw_df) else 0,
    "missing_values_in_raw_file": {
        column: int(raw_df[column].isna().sum()) for column in raw_df.columns
    },
    "rejection_reason_counts": rejection_counts
}
print(json.dumps(quality_summary, indent=4))

{
    "source_file": "raw_orders.csv",
    "total_records": 253,
    "valid_records": 149,
    "rejected_records": 104,
    "rejection_rate_percent": 41.11,
    "missing_values_in_raw_file": {
        "order_id": 0,
        "customer_id": 10,
        "order_date": 0,
        "quantity": 0,
        "unit_price": 6,
        "sales_region": 41,
        "payment_status": 41
    },
    "rejection_reason_counts": {
        "missing_order_id": 0,
        "missing_customer_id": 10,
        "invalid_order_date": 8,
        "invalid_quantity": 14,
        "invalid_unit_price": 11,
        "missing_sales_region": 41,
        "missing_payment_status": 41,
        "exact_duplicate": 2
    }
}


## 10. Export outputs

In [13]:
valid_df.to_csv(OUTPUT_DIR / "cleaned_data.csv", index=False)
rejected_df.to_csv(OUTPUT_DIR / "rejected_data.csv", index=False)

with open(OUTPUT_DIR / "quality_report.json", "w", encoding="utf-8") as file:
    json.dump(quality_summary, file, indent=4)

print("Exported to:", OUTPUT_DIR)

Exported to: C:\jn\month 1 project\AI_Data_Quality_Engine\output


## 11. Gemini AI analysis 

In [19]:
import os
import json
from google import genai
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("Gemini skipped: GEMINI_API_KEY environment variable is not set.")
else:
    client = genai.Client(api_key=api_key)

    prompt = f"""
You are a data quality analyst. Interpret this aggregate report.
Explain overall quality, largest rejection categories, plausible upstream causes
(label causes as hypotheses), and practical recommendations.
Do not invent metrics. Do not recommend silently deleting rejected records.

REPORT:
{json.dumps(quality_summary, indent=2)}
"""

    response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=prompt
    )
    print(response.text)
    client.close()

Enter your Gemini API key:  ········


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

## 12. Final reconciliation checks

In [18]:
assert len(raw_df) == len(valid_df) + len(rejected_df)
assert valid_df["rejected"].sum() == 0
assert rejected_df["rejected"].all()
print("All pipeline checks passed.")

All pipeline checks passed.
